# Smoke Detector — Kaggle Tur 2 Eğitimi (Pyro-SDIS + D-Fire)

Sprint 2 Parça (b) tur 2 — Kaggle P100/T4 üstünde full baseline eğitimi.

## Hedef
Val ROC-AUC ≥ 0.82 (tur 1: 0.66 — class imbalance + checkpoint kriteri yanlış).

## Tur 1'den farklar (architect kararı)
- `BCEWithLogitsLoss(pos_weight=n_neg/n_pos)` — dinamik class imbalance dengelemesi
- `epochs=20`, `freeze_epochs=5`
- BEST checkpoint kriteri F1 değil **ROC-AUC**
- `RandomResizedCrop(scale=(0.6, 1.0))` — daha agresif crop
- Her 5 epoch'ta intermediate checkpoint (Kaggle 12 sa session reservi)
- D-Fire dahil (tur 1 sadece Pyro-SDIS idi — D-Fire indirilmemişti)

## Kaggle setup
1. **Settings → Accelerator**: `GPU P100` (16 GB tek GPU) **veya** `GPU T4 x2` (T4 tek başına da çalışır; multi-GPU kullanmıyoruz).
2. **Settings → Internet**: `ON` (HuggingFace + pip için zorunlu).
3. **Add Data** (sağ panel) → ara: `gaiasd/dfire-dataset` → ekle. Mount path: `/kaggle/input/dfire-dataset/`.
4. **Save Version → Save & Run All (Commit)** komitte 12 sa GPU quota geçerli. Interactive editöründe 9 sa.

## Çıktı
Eğitim sonu `best_checkpoint.pt` + `training_summary.json` `/kaggle/working/` altına kopyalanır. Save Version sonrası **Output** sekmesinden indirilebilir.

## Hücre 2 — Repo klonla ve bağımlılıkları kur

Kaggle'da `torch` zaten kurulu (önceden derlenmiş CUDA wheel). Sadece eksikleri (albumentations, datasets v2.19+, sklearn varyantları) ekleriz. `pyproject.toml` tool config olduğu için `pip install -e .` çalışmaz — `requirements.txt` kullanırız.

In [ ]:
!git clone https://github.com/mkupeli/wildfire-ai-ml.git
%cd wildfire-ai-ml
!git log -1 --oneline

In [ ]:
# torch + torchvision Kaggle imajında zaten kurulu — atla.
# Asıl ihtiyacımız: albumentations, datasets, pyarrow, scikit-learn (zaten var ama versiyon).
!pip install -q "albumentations>=1.4" "datasets>=2.19" "pyarrow>=15" "huggingface_hub>=0.23"
!pip install -q "opencv-python-headless>=4.8"

## Hücre 3 — D-Fire dataset bağla

Kaggle Add Data ile mount edilen `/kaggle/input/dfire-dataset/` repo içindeki `data/raw/dfire/` ile uyumlu hale getirilir.

**Önce yapıyı keşfet** (Kaggle dataset yapısı zaman içinde değişebilir, FireDataset `data/raw/dfire/images/{split}/` ve `data/raw/dfire/labels/{split}/` bekler, split ∈ {`train`, `val`}).

In [ ]:
import os
from pathlib import Path

DFIRE_SRC = Path("/kaggle/input/dfire-dataset")
print("--- D-Fire root listing ---")
for p in sorted(DFIRE_SRC.iterdir()):
    print(p)
print()
# 2-3 seviye derinliği göster
for root, dirs, files in os.walk(DFIRE_SRC):
    depth = len(Path(root).relative_to(DFIRE_SRC).parts)
    if depth > 2:
        continue
    print(f"[d={depth}] {root} (dirs={len(dirs)}, files={len(files)})")
    if depth == 2 and files[:3]:
        for f in files[:3]:
            print(f"   sample: {f}")

In [ ]:
# D-Fire Kaggle yapısı tipik olarak:
#   /kaggle/input/dfire-dataset/train/images/*.jpg
#   /kaggle/input/dfire-dataset/train/labels/*.txt
#   /kaggle/input/dfire-dataset/test/images/*.jpg
#   /kaggle/input/dfire-dataset/test/labels/*.txt
#
# FireDataset şu yapıyı bekler (src/wildfire_ml/data/dataset.py _load_dfire):
#   data/raw/dfire/images/{train,val}/*.jpg
#   data/raw/dfire/labels/{train,val}/*.txt
#
# Mapping: Kaggle 'test' -> bizim 'val' (D-Fire'da resmi val ayrımı yok).
# Symlink ile kopya yok, disk israfı sıfır.

import os
from pathlib import Path

DFIRE_SRC = Path("/kaggle/input/dfire-dataset")
DFIRE_DST = Path("data/raw/dfire")
DFIRE_DST.mkdir(parents=True, exist_ok=True)
(DFIRE_DST / "images").mkdir(exist_ok=True)
(DFIRE_DST / "labels").mkdir(exist_ok=True)

# Olası kaynak alt-dizinleri otomatik tespit et
candidates = {
    "train": [DFIRE_SRC / "train", DFIRE_SRC / "D-Fire" / "train"],
    "val":   [DFIRE_SRC / "test",  DFIRE_SRC / "D-Fire" / "test"],
}

for target_split, srcs in candidates.items():
    src_base = next((s for s in srcs if s.exists()), None)
    if src_base is None:
        print(f"[WARN] {target_split} için kaynak bulunamadı — Hücre 3 (ls çıktısı) ile manuel ayarla")
        continue
    for kind in ("images", "labels"):
        src = src_base / kind
        dst = DFIRE_DST / kind / target_split
        if dst.exists() or dst.is_symlink():
            print(f"[skip] {dst} zaten var")
            continue
        if not src.exists():
            print(f"[WARN] {src} yok — atlandı")
            continue
        os.symlink(src, dst, target_is_directory=True)
        print(f"[link] {dst} -> {src}")

# Doğrulama: kaç dosya görünüyor?
for split in ("train", "val"):
    img_dir = DFIRE_DST / "images" / split
    lbl_dir = DFIRE_DST / "labels" / split
    n_img = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"[dfire/{split}] images={n_img} labels={n_lbl}")

## Hücre 4 — Pyro-SDIS indir (HuggingFace)

FireDataset Pyro-SDIS'i `data/raw/pyro-sdis/pyronear___pyro-sdis/default/0.0.0/<hash>/` yapısında arar (HuggingFace `datasets` v3 cache layout). `download_pyrosdis.py` `load_dataset(..., cache_dir=...)` ile bunu otomatik kurar.

In [ ]:
!python scripts/download_pyrosdis.py --output data/raw/pyro-sdis

In [ ]:
# Sağlık kontrolü: dataset.py beklediği hash dizini var mı?
from pathlib import Path
cache = Path("data/raw/pyro-sdis/pyronear___pyro-sdis/default/0.0.0")
if cache.exists():
    for sub in cache.iterdir():
        print(sub)
        for f in sorted(sub.glob("*.arrow")):
            print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"[WARN] {cache} yok — download_pyrosdis çıktısını kontrol et")

# Bilgi: dataset.py'da hash sabit kodlu:
#   a1e553ec4d806f71fc6db744cc22bc3469487382
# Yeni dataset revizyonu farklı hash dönerse FireDataset onu görmez.
# Eğer farklı hash inerse symlink ile sabit hash'e bağla:
EXPECTED = "a1e553ec4d806f71fc6db744cc22bc3469487382"
if cache.exists():
    actual_dirs = [d.name for d in cache.iterdir() if d.is_dir()]
    if EXPECTED not in actual_dirs and actual_dirs:
        import os
        actual = cache / actual_dirs[0]
        target = cache / EXPECTED
        if not target.exists():
            os.symlink(actual, target, target_is_directory=True)
            print(f"[link] {target} -> {actual} (hash uyumsuzluğu düzeltildi)")

In [ ]:
# Son kontrol: FireDataset başarıyla yükleniyor mu? (Birkaç saniye sürer.)
import sys
sys.path.insert(0, "src")
from pathlib import Path
from wildfire_ml.data.dataset import FireDataset
from wildfire_ml.data.transforms import build_transforms

train_ds = FireDataset(Path("data/raw"), "train", build_transforms("train"))
val_ds = FireDataset(Path("data/raw"), "val", build_transforms("val"))
n_pos_train = sum(1 for _, l in train_ds.samples if l == 1)
n_pos_val = sum(1 for _, l in val_ds.samples if l == 1)
print(f"train: total={len(train_ds)} pos={n_pos_train} neg={len(train_ds) - n_pos_train}")
print(f"val:   total={len(val_ds)} pos={n_pos_val} neg={len(val_ds) - n_pos_val}")
print(f"pos_weight (train) = {(len(train_ds) - n_pos_train) / max(n_pos_train, 1):.4f}")

## Hücre 5 — Eğitimi başlat

Tur 2 parametreleri: `--epochs 20 --freeze-epochs 5 --batch-size 64`. P100 16 GB için batch 64 güvenli; OOM olursa 32'ye düş, VRAM bolsa 128 dene. Kaggle Linux'ta `num_workers=2` güvenli (Windows tek thread zorunluluğu yok).

`-u` stdout buffer kapalı — log canlı akar.

In [ ]:
!python -u scripts/train_smoke_detector.py \
    --device cuda \
    --batch-size 64 \
    --num-workers 2 \
    --epochs 20 \
    --freeze-epochs 5 \
    --checkpoint-every 5

## Hücre 6 — Çıktıları `/kaggle/working/` altına taşı

Kaggle session sonu **sadece `/kaggle/working/`** altındaki dosyalar Output dataset'e dahil edilir. Repo klonladığımız `wildfire-ai-ml/` `/kaggle/working/wildfire-ai-ml/` altında zaten — ama dolaylı yoldan içeride. Doğrudan output kökünde olsun ki **Save Version** sonrası kolayca indirilebilsin.

In [ ]:
import shutil
from pathlib import Path

src = Path("artifacts/full_train")
dst = Path("/kaggle/working/full_train_tur2")
dst.mkdir(parents=True, exist_ok=True)

for fname in ("best_checkpoint.pt", "training_summary.json"):
    s = src / fname
    if s.exists():
        shutil.copy2(s, dst / fname)
        print(f"[copy] {s} -> {dst / fname} ({s.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"[WARN] {s} yok — eğitim tamamlanmamış olabilir")

# Intermediate checkpoint'leri de kopyala (Save Version 20 GB output limiti var; sıkışırsa atla)
for cp in src.glob("checkpoint_epoch*.pt"):
    shutil.copy2(cp, dst / cp.name)
    print(f"[copy] {cp.name} ({cp.stat().st_size / 1e6:.2f} MB)")

print("\n--- /kaggle/working/full_train_tur2 içeriği ---")
for f in sorted(dst.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Hızlı özet — training_summary.json'dan BEST metrik
import json
from pathlib import Path

summary_path = Path("/kaggle/working/full_train_tur2/training_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    best = summary.get("best_metrics", {})
    cfg = summary.get("config", {})
    print(f"epochs={cfg.get('epochs')} freeze_epochs={cfg.get('freeze_epochs')} batch={cfg.get('batch_size')}")
    print(f"pos_weight={cfg.get('pos_weight'):.4f}" if cfg.get('pos_weight') else "pos_weight=?")
    print(f"BEST  ROC-AUC={best.get('roc_auc', 0):.4f}  F1={best.get('f1', 0):.4f}  P={best.get('precision', 0):.4f}  R={best.get('recall', 0):.4f}  acc={best.get('accuracy', 0):.4f}")
    print(f"CM={best.get('confusion_matrix')}")
    print(f"\nHedef ROC-AUC >= 0.82  ->  {'PASS' if best.get('roc_auc', 0) >= 0.82 else 'FAIL'}")